In [98]:
import pandas as pd

In [99]:
tracking = pd.concat([pd.read_parquet("data/nhl_event_data_20232024.parquet"),
                      pd.read_parquet("data/nhl_event_data_20242025.parquet"),
                      pd.read_parquet("data/nhl_event_data_20252026.parquet")])
pbp = pd.concat([pd.read_csv("data/pbp_20232024.csv"),pd.read_csv("data/pbp_20242025.csv"),pd.read_csv("data/pbp_20252026.csv")])
pbp = pbp[pbp["situationCode"]==1551]
merged = pd.merge(tracking, pbp[["gameId","eventId","xG", "periodNumber","secondsElapsedInPeriod","isHome","eventOwnerTeamId"]], left_on=["game_id","ev_id"],right_on=["gameId","eventId"])

/var/folders/7y/fwm3j5vs3p72s234bsqbz8sw0000gp/T/ipykernel_17513/731721272.py:4: DtypeWarning: Columns (16: homeIsEmptyNet, 17: awayIsEmptyNet) have mixed types. Specify dtype option on import or set low_memory=False.
  pbp = pd.concat([pd.read_csv("data/pbp_20232024.csv"),pd.read_csv("data/pbp_20242025.csv"),pd.read_csv("data/pbp_20252026.csv")])


In [100]:
import re
import numpy as np
import pandas as pd



def normalize_direction(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()


    slot_re = re.compile(r"^(home|away)_on_ice_(\d+)_")

    # true on-ice counts, the dataset rejects frames with more than 6 and the slots that show it are dropped below
    for side in ["home", "away"]:
        side_ids = [c for c in df.columns if re.fullmatch(rf"{side}_on_ice_\d+_id", c)]
        df[f"{side}_n_on_ice"] = df[side_ids].notna().sum(axis=1)

    drop = [c for c in df.columns if (m := slot_re.match(c)) and int(m.group(2)) >= 7]
    df = df.drop(columns=drop)



    df = df.sort_values(["game_id", "ev_id", "time_stamp"])
    attack_dir = np.sign(df.groupby(["game_id", "ev_id"])["puck_x"].transform("last"))
    df["attack_dir"] = attack_dir

    x_cols = [c for c in df.columns if c.endswith("_x")]
    y_cols = [c for c in df.columns if c.endswith("_y")]
    df[x_cols] = df[x_cols].mul(attack_dir, axis=0)
    df[y_cols] = df[y_cols].mul(attack_dir, axis=0)
    return df

merged = normalize_direction(merged)

In [101]:
keys = ["gameId", "periodNumber", "eventOwnerTeamId"]


pbp = pbp.sort_values(["gameId", "sortOrder"])
is_stop = pbp["eventTypeDescKey"].isin(["faceoff", "stoppage", "period-start"])
pbp["seq_id"] = is_stop.groupby(pbp["gameId"]).cumsum()

goals = (pbp.loc[pbp["eventTypeDescKey"] == "goal", keys + ["secondsElapsedInPeriod", "seq_id"]]
            .rename(columns={"secondsElapsedInPeriod": "goal_t", "seq_id": "goal_seq_id"})
            .sort_values("goal_t"))

pbp = pd.merge_asof(pbp.sort_values("secondsElapsedInPeriod"), goals,
                    left_on="secondsElapsedInPeriod", right_on="goal_t",
                    by=keys, direction="forward")

pbp = pbp[(pbp["eventTypeDescKey"] == "goal") |
          ((pbp["goal_t"] - pbp["secondsElapsedInPeriod"] < 10) & (pbp["seq_id"] == pbp["goal_seq_id"]))]
pbp = pbp[pbp["eventTypeDescKey"].isin(["goal","shot-on-goal","blocked-shot","missed-shot"])]

/var/folders/7y/fwm3j5vs3p72s234bsqbz8sw0000gp/T/ipykernel_17513/1279688949.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pbp["seq_id"] = is_stop.groupby(pbp["gameId"]).cumsum()


In [102]:
GOAL_LINE_X, NET_HALF_WIDTH, NET_DEPTH = 89, 3, 3.5   # goal line 11 ft in from the end boards (x=100); net ~6 ft wide, ~3.5 ft deep
SHOT_SPEED = 30 * 5280 / 3600         # 30 mph in ft/s

merged = merged.sort_values(["game_id", "ev_id", "time_stamp"])
clip = merged.groupby(["game_id", "ev_id"])

# puck velocity in ft/s; time_stamp is in 0.1s frames, duplicate timestamps get NaN
dt = clip["time_stamp"].diff().replace(0, np.nan) / 10
merged["puck_vx"] = clip["puck_x"].diff() / dt
merged["puck_vy"] = clip["puck_y"].diff() / dt
merged["puck_speed"] = np.hypot(merged["puck_vx"], merged["puck_vy"])

# goal frame = first frame the puck is inside the net (past the goal line, between the posts, not behind the net)
in_net = (merged["puck_x"].between(GOAL_LINE_X, GOAL_LINE_X + NET_DEPTH)
          & (merged["puck_y"].abs() <= NET_HALF_WIDTH))
merged["goal_ts"] = merged["time_stamp"].where(in_net).groupby([merged["game_id"], merged["ev_id"]]).transform("min")

print("clips where the puck never enters the net (dropped):",
      merged.loc[merged["goal_ts"].isna(), ["game_id", "ev_id"]].drop_duplicates().shape[0])
merged = merged[merged["time_stamp"] <= merged["goal_ts"]].reset_index(drop=True)

# seconds elapsed in period for each frame, counted back from the goal's pbp time
merged["frame_sec"] = merged["secondsElapsedInPeriod"] - (merged["goal_ts"] - merged["time_stamp"]) / 10

clips where the puck never enters the net (dropped): 327


In [103]:
# match each pbp shot to the frame where the puck is closest to (and within 3 ft of) its recorded spot
shots = (pbp.loc[pbp["eventTypeDescKey"] != "goal", ["gameId", "periodNumber", "goal_t", "eventId", "secondsElapsedInPeriod", "xCoord", "yCoord", "xG"]]
            .rename(columns={"eventId": "shot_ev_id", "secondsElapsedInPeriod": "shot_t"}))
cand = merged[["game_id", "ev_id", "periodNumber", "secondsElapsedInPeriod", "time_stamp", "frame_sec", "puck_x", "puck_y", "attack_dir"]].merge(
    shots, left_on=["game_id", "periodNumber", "secondsElapsedInPeriod"], right_on=["gameId", "periodNumber", "goal_t"])
n_shots = cand[["game_id", "ev_id", "shot_ev_id"]].drop_duplicates().shape[0]

# pbp times are whole seconds (and so is the goal time we count back from), and the puck tends to
# reach the shot spot ~1s after the pbp time, so allow 1s before to 2s after
cand = cand[(cand["frame_sec"] - cand["shot_t"]).between(-1, 2, inclusive="neither")]
# pbp y is mirrored relative to tracking y
cand["dist"] = np.hypot(cand["puck_x"] - cand["xCoord"] * cand["attack_dir"],
                        cand["puck_y"] + cand["yCoord"] * cand["attack_dir"])
cand = cand[cand["dist"] <= 3]
shot_frames = cand.loc[cand.groupby(["game_id", "ev_id", "shot_ev_id"])["dist"].idxmin(),
                       ["game_id", "ev_id", "shot_ev_id", "time_stamp", "xG"]].sort_values("time_stamp")
print(f"matched {len(shot_frames)} of {n_shots} shots in clips")

# frames up to the end of the shot's >SHOT_SPEED burst belong to that shot; everything after goes to the next shot / the goal
merged["seq_ev_id"] = merged["ev_id"]
ts_all, speed_all = merged["time_stamp"].to_numpy(), merged["puck_speed"].to_numpy()
rows_by_clip = merged.groupby(["game_id", "ev_id"]).indices
for key, s in shot_frames.groupby(["game_id", "ev_id"]):
    rows = rows_by_clip[key]
    ts, speed = ts_all[rows], speed_all[rows]
    shot_ts, start = s["time_stamp"].to_numpy(), 0
    for k, shot_id in enumerate(s["shot_ev_id"]):
        i = np.searchsorted(ts, shot_ts[k])
        nxt = np.searchsorted(ts, shot_ts[k + 1]) if k + 1 < len(shot_ts) else len(ts)
        fast = np.flatnonzero(speed[i:nxt] > SHOT_SPEED)
        end = i
        if len(fast):
            slow = np.flatnonzero(speed[i + fast[0]:nxt] <= SHOT_SPEED)
            end = i + fast[0] + (slow[0] if len(slow) else nxt - i - fast[0]) - 1
        merged.loc[rows[start:end + 1], "seq_ev_id"] = shot_id
        start = end + 1

merged["is_goal"] = (merged["seq_ev_id"] == merged["ev_id"]).astype(int)

# release frame of the goal itself: same matching against the goal's pbp spot, falling back to the frame it enters the net
g = merged[["game_id", "ev_id", "time_stamp", "frame_sec", "secondsElapsedInPeriod", "puck_x", "puck_y", "attack_dir"]].merge(
    pbp.loc[pbp["eventTypeDescKey"] == "goal", ["gameId", "eventId", "xCoord", "yCoord"]],
    left_on=["game_id", "ev_id"], right_on=["gameId", "eventId"])
g = g[(g["frame_sec"] - g["secondsElapsedInPeriod"]).between(-1, 2, inclusive="neither")]
g["dist"] = np.hypot(g["puck_x"] - g["xCoord"] * g["attack_dir"], g["puck_y"] + g["yCoord"] * g["attack_dir"])
g = g[g["dist"] <= 3]
goal_frames = g.loc[g.groupby(["game_id", "ev_id"])["dist"].idxmin(), ["game_id", "ev_id", "time_stamp"]]
print(f"matched release frame for {len(goal_frames)} of {merged[['game_id', 'ev_id']].drop_duplicates().shape[0]} goals")

# xG of the shot each frame belongs to, and seconds until that shot's release (negative once the puck is released)
release = pd.concat([shot_frames.rename(columns={"shot_ev_id": "seq_ev_id"}),
                     goal_frames.assign(seq_ev_id=goal_frames["ev_id"])]).rename(columns={"time_stamp": "release_ts", "xG": "shot_xG"})
merged = merged.merge(release, on=["game_id", "ev_id", "seq_ev_id"], how="left")
merged["release_ts"] = merged["release_ts"].fillna(merged["goal_ts"].where(merged["is_goal"] == 1))
merged["shot_xG"] = merged["shot_xG"].fillna(merged["xG"].where(merged["is_goal"] == 1))
merged["time_to_shot"] = (merged["release_ts"] - merged["time_stamp"]) / 10




matched 3633 of 4614 shots in clips
matched release frame for 11959 of 14182 goals


In [104]:
merged["shot_id"] = merged.apply(lambda row: f"{row["game_id"]}_{row["seq_ev_id"]}",axis=1)

size = merged.groupby("shot_id").agg(
    frames=("shot_id", "count"),
)

metadata = pd.read_csv("data/metadata.csv")
pos = metadata.set_index("playerId")["position"]

slot_cols = {}
for side in ["home", "away"]:
    for n in range(1, 7):
        slot = f"{side}_on_ice_{n}"
        slot_cols[f"{slot}_pos"] = merged[f"{slot}_id"].map(pos).fillna("G").where(merged[f"{slot}_id"].notna())
        slot_cols[f"{slot}_off"] = merged[f"{slot}_team_id"] == merged["eventOwnerTeamId"]
merged = pd.concat([merged, pd.DataFrame(slot_cols, index=merged.index)], axis=1)

In [105]:
size["frames"].median()

np.float64(82.0)

In [106]:
merged = merged.merge(size, on="shot_id")
merged = merged[merged["frames"] >= 50] # minimum of 5 seconds per
merged = merged[merged["frames"] <= 100]

In [107]:
merged.to_csv("data/processed_ppt.csv")

In [108]:
merged

,time_stamp,home_on_ice_1_tracking_id,home_on_ice_1_id,home_on_ice_1_x,home_on_ice_1_y,home_on_ice_1_sweater_number,home_on_ice_1_team_id,home_on_ice_1_team_abbr,home_on_ice_2_tracking_id,home_on_ice_2_id,...,away_on_ice_2_off,away_on_ice_3_pos,away_on_ice_3_off,away_on_ice_4_pos,away_on_ice_4_off,away_on_ice_5_pos,away_on_ice_5_off,away_on_ice_6_pos,away_on_ice_6_off,frames
0,16969742463,14021.0,8478010.0,-22.941133,-11.190208,21.0,14.0,TBL,14031.0,8477992.0,...,False,L,False,D,False,G,False,C,False,83
1,16969742464,14021.0,8478010.0,-21.068550,-12.695908,21.0,14.0,TBL,14031.0,8477992.0,...,False,L,False,D,False,G,False,C,False,83
2,16969742465,14021.0,8478010.0,-19.188283,-14.147296,21.0,14.0,TBL,14031.0,8477992.0,...,False,L,False,D,False,G,False,C,False,83
3,16969742466,14021.0,8478010.0,-17.302133,-15.562960,21.0,14.0,TBL,14031.0,8477992.0,...,False,L,False,D,False,G,False,C,False,83
4,16969742467,14021.0,8478010.0,-15.393025,-16.966641,21.0,14.0,TBL,14031.0,8477992.0,...,False,L,False,D,False,G,False,C,False,83
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1281662,17814880566,54004.0,8478397.0,79.602833,-11.576098,4.0,54.0,VGK,54015.0,8478396.0,...,True,D,True,G,True,R,True,L,True,66
1281663,17814880567,54004.0,8478397.0,79.128642,-11.608635,4.0,54.0,VGK,54015.0,8478396.0,...,True,D,True,G,True,R,True,L,True,66
1281664,17814880568,54004.0,8478397.0,78.751392,-11.464378,4.0,54.0,VGK,54015.0,8478396.0,...,True,D,True,G,True,R,True,L,True,66
1281665,17814880569,54004.0,8478397.0,78.475708,-11.139248,4.0,54.0,VGK,54015.0,8478396.0,...,True,D,True,G,True,R,True,L,True,66
